# VASTRA Pro backend — OOTDiffusion (mask-accurate try-on + fit control)

**Run all cells top to bottom, then copy the `gradio.live` link and open:**
`index.html?backend=PASTE-LINK-HERE`

What makes this "Pro":
- **Mask-accurate:** the AI repaints ONLY the garment area (upper / lower / dress).
- **Body preservation:** after generation, your ORIGINAL face, skin, arms, legs and background pixels are pasted back — bit-exact, zero change.
- **Fit control:** `slim / regular / loose` grows the repaint mask so the same outfit renders tighter or roomier.

Needs: GPU T4 x2 + Internet ON. Time per try-on ≈ 1.5–2.5 min on free T4.

In [ ]:
# Cell 2 — one-time setup (~4-6 min). Run once per session.
import os
print('cwd:', os.getcwd())
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q diffusers==0.24.0 transformers==4.36.2 accelerate==0.26.1 onnxruntime einops scikit-image config huggingface_hub safetensors
!pip install -q -U gradio
import torch, diffusers, transformers, gradio
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('diffusers', diffusers.__version__, '| transformers', transformers.__version__, '| gradio', gradio.__version__)

In [ ]:
# Cell 3 — code + weights download (~5-8 min first time, cached after).
import os
WORK = '/kaggle/working' if os.path.isdir('/kaggle') else '/content'
print('WORK =', WORK)
os.chdir(WORK)
if not os.path.isdir(f'{WORK}/OOTDiffusion'):
    !git clone --depth 1 https://github.com/levihsu/OOTDiffusion
from huggingface_hub import snapshot_download
if not os.path.isdir(f'{WORK}/OOTDiffusion/checkpoints/ootd'):
    snapshot_download('levihsu/OOTDiffusion', allow_patterns=['checkpoints/*'], local_dir=f'{WORK}/ootd_weights')
    !rm -rf {WORK}/OOTDiffusion/checkpoints
    !mv {WORK}/ootd_weights/checkpoints {WORK}/OOTDiffusion/checkpoints
if not os.path.isdir(f'{WORK}/OOTDiffusion/checkpoints/clip-vit-large-patch14'):
    snapshot_download('openai/clip-vit-large-patch14', local_dir=f'{WORK}/OOTDiffusion/checkpoints/clip-vit-large-patch14')
!ls {WORK}/OOTDiffusion/checkpoints
!du -sh {WORK}/OOTDiffusion/checkpoints

In [ ]:
# Cell 4 — load models + start server. KEEP THIS TAB OPEN, copy the gradio.live URL.
import gc, os, sys, time
import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw
import gradio as gr

WORK = '/kaggle/working' if os.path.isdir('/kaggle') else '/content'
os.chdir(f'{WORK}/OOTDiffusion/run')  # OOTD uses relative ../checkpoints paths
sys.path.insert(0, f'{WORK}/OOTDiffusion')
from preprocess.openpose.run_openpose import OpenPose
from preprocess.humanparsing.run_parsing import Parsing
from ootd.inference_ootd_dc import OOTDiffusionDC

# ---------- config (T4-safe; HIGH_RES=True only on bigger GPUs) ----------
HIGH_RES = False
W, H = (768, 1024) if HIGH_RES else (576, 768)
STEPS, SEED = 25, 42
CAT_MODEL = {'upper': 'upperbody', 'lower': 'lowerbody', 'dress': 'dress'}
CAT_MASK = {'upper': 'upper_body', 'lower': 'lower_body', 'dress': 'dresses'}

# ---------- mask helpers (vendored from OOTDiffusion run/utils_ootd.py, CC BY-NC-SA 4.0) ----------
label_map = {'background': 0, 'hat': 1, 'hair': 2, 'sunglasses': 3, 'upper_clothes': 4,
             'skirt': 5, 'pants': 6, 'dress': 7, 'belt': 8, 'left_shoe': 9, 'right_shoe': 10,
             'head': 11, 'left_leg': 12, 'right_leg': 13, 'left_arm': 14, 'right_arm': 15,
             'bag': 16, 'scarf': 17}


def extend_arm_mask(wrist, elbow, scale):
    wrist = elbow + scale * (wrist - elbow)
    return wrist


def hole_fill(img):
    img = np.pad(img[1:-1, 1:-1], pad_width=1, mode='constant', constant_values=0)
    img_copy = img.copy()
    mask = np.zeros((img.shape[0] + 2, img.shape[1] + 2), dtype=np.uint8)
    cv2.floodFill(img, mask, (0, 0), 255)
    img_inverse = cv2.bitwise_not(img)
    dst = cv2.bitwise_or(img_copy, img_inverse)
    return dst


def refine_mask(mask):
    contours, hierarchy = cv2.findContours(mask.astype(np.uint8), cv2.RETR_CCOMP, cv2.CHAIN_APPROX_TC89_L1)
    area = []
    for j in range(len(contours)):
        a_d = cv2.contourArea(contours[j], True)
        area.append(abs(a_d))
    refine_mask = np.zeros_like(mask).astype(np.uint8)
    if len(area) != 0:
        i = area.index(max(area))
        cv2.drawContours(refine_mask, contours, i, color=255, thickness=-1)
    return refine_mask


FIT_DILATE = {'slim': 2, 'regular': 5, 'loose': 9}
FIT_ARM_ADJ = {'slim': -8, 'regular': 0, 'loose': 10}
FIT_BOTTOM_MARGIN = {'slim': 8, 'regular': 20, 'loose': 32}


def get_mask_location(model_type, category, model_parse, keypoint, fit='regular', width=384, height=512):
    im_parse = model_parse.resize((width, height), Image.NEAREST)
    parse_array = np.array(im_parse)
    if model_type == 'hd':
        arm_width = 60
    elif model_type == 'dc':
        arm_width = 45
    else:
        raise ValueError("model_type must be 'hd' or 'dc'!")
    arm_width += FIT_ARM_ADJ.get(fit, 0)
    parse_head = (parse_array == 1).astype(np.float32) + (parse_array == 3).astype(np.float32) + (parse_array == 11).astype(np.float32)
    parser_mask_fixed = (parse_array == label_map['left_shoe']).astype(np.float32) + (parse_array == label_map['right_shoe']).astype(np.float32) + (parse_array == label_map['hat']).astype(np.float32) + (parse_array == label_map['sunglasses']).astype(np.float32) + (parse_array == label_map['bag']).astype(np.float32)
    parser_mask_changeable = (parse_array == label_map['background']).astype(np.float32)
    arms_left = (parse_array == 14).astype(np.float32)
    arms_right = (parse_array == 15).astype(np.float32)
    if category == 'dresses':
        parse_mask = (parse_array == 7).astype(np.float32) + (parse_array == 4).astype(np.float32) + (parse_array == 5).astype(np.float32) + (parse_array == 6).astype(np.float32)
        parser_mask_changeable += np.logical_and(parse_array, np.logical_not(parser_mask_fixed))
    elif category == 'upper_body':
        parse_mask = (parse_array == 4).astype(np.float32) + (parse_array == 7).astype(np.float32)
        parser_mask_fixed += (parse_array == label_map['skirt']).astype(np.float32) + (parse_array == label_map['pants']).astype(np.float32)
        parser_mask_changeable += np.logical_and(parse_array, np.logical_not(parser_mask_fixed))
    elif category == 'lower_body':
        parse_mask = (parse_array == 6).astype(np.float32) + (parse_array == 12).astype(np.float32) + (parse_array == 13).astype(np.float32) + (parse_array == 5).astype(np.float32)
        parser_mask_fixed += (parse_array == label_map['upper_clothes']).astype(np.float32) + (parse_array == 14).astype(np.float32) + (parse_array == 15).astype(np.float32)
        parser_mask_changeable += np.logical_and(parse_array, np.logical_not(parser_mask_fixed))
    else:
        raise NotImplementedError
    pose_data = np.array(keypoint['pose_keypoints_2d']).reshape((-1, 2))
    im_arms_left = Image.new('L', (width, height))
    im_arms_right = Image.new('L', (width, height))
    arms_draw_left = ImageDraw.Draw(im_arms_left)
    arms_draw_right = ImageDraw.Draw(im_arms_right)
    if category == 'dresses' or category == 'upper_body':
        shoulder_right = np.multiply(tuple(pose_data[2][:2]), height / 512.0)
        shoulder_left = np.multiply(tuple(pose_data[5][:2]), height / 512.0)
        elbow_right = np.multiply(tuple(pose_data[3][:2]), height / 512.0)
        elbow_left = np.multiply(tuple(pose_data[6][:2]), height / 512.0)
        wrist_right = np.multiply(tuple(pose_data[4][:2]), height / 512.0)
        wrist_left = np.multiply(tuple(pose_data[7][:2]), height / 512.0)
        ARM_LINE_WIDTH = int(arm_width / 512 * height)
        size_left = [shoulder_left[0] - ARM_LINE_WIDTH // 2, shoulder_left[1] - ARM_LINE_WIDTH // 2, shoulder_left[0] + ARM_LINE_WIDTH // 2, shoulder_left[1] + ARM_LINE_WIDTH // 2]
        size_right = [shoulder_right[0] - ARM_LINE_WIDTH // 2, shoulder_right[1] - ARM_LINE_WIDTH // 2, shoulder_right[0] + ARM_LINE_WIDTH // 2, shoulder_right[1] + ARM_LINE_WIDTH // 2]
        if wrist_right[0] <= 1. and wrist_right[1] <= 1.:
            im_arms_right = arms_right
        else:
            wrist_right = extend_arm_mask(wrist_right, elbow_right, 1.2)
            arms_draw_right.line(np.concatenate((shoulder_right, elbow_right, wrist_right)).astype(np.uint16).tolist(), 'white', ARM_LINE_WIDTH, 'curve')
            arms_draw_right.arc(size_right, 0, 360, 'white', ARM_LINE_WIDTH // 2)
        if wrist_left[0] <= 1. and wrist_left[1] <= 1.:
            im_arms_left = arms_left
        else:
            wrist_left = extend_arm_mask(wrist_left, elbow_left, 1.2)
            arms_draw_left.line(np.concatenate((wrist_left, elbow_left, shoulder_left)).astype(np.uint16).tolist(), 'white', ARM_LINE_WIDTH, 'curve')
            arms_draw_left.arc(size_left, 0, 360, 'white', ARM_LINE_WIDTH // 2)
        hands_left = np.logical_and(np.logical_not(im_arms_left), arms_left)
        hands_right = np.logical_and(np.logical_not(im_arms_right), arms_right)
        parser_mask_fixed += hands_left + hands_right
    parser_mask_fixed = np.logical_or(parser_mask_fixed, parse_head)
    parse_mask = cv2.dilate(parse_mask, np.ones((5, 5), np.uint16), iterations=FIT_DILATE.get(fit, 5))  # ★ FIT HOOK
    if category == 'dresses' or category == 'upper_body':
        neck_mask = (parse_array == 18).astype(np.float32)
        neck_mask = cv2.dilate(neck_mask, np.ones((5, 5), np.uint16), iterations=1)
        neck_mask = np.logical_and(neck_mask, np.logical_not(parse_head))
        parse_mask = np.logical_or(parse_mask, neck_mask)
        arm_mask = cv2.dilate(np.logical_or(im_arms_left, im_arms_right).astype('float32'), np.ones((5, 5), np.uint16), iterations=4)
        parse_mask += np.logical_or(parse_mask, arm_mask)
    parse_mask = np.logical_and(parser_mask_changeable, np.logical_not(parse_mask))
    parse_mask_total = np.logical_or(parse_mask, parser_mask_fixed)
    inpaint_mask = 1 - parse_mask_total
    img = np.where(inpaint_mask, 255, 0)
    dst = hole_fill(img.astype(np.uint8))
    dst = refine_mask(dst)
    inpaint_mask = dst / 255 * 1
    if fit == 'loose':
        inpaint_mask = cv2.dilate((inpaint_mask * 255).astype(np.uint8), np.ones((5, 5), np.uint8), iterations=2).astype(np.float32) / 255.0
    if category == 'upper_body':
        # Mask must not run far below the original garment's hem (white-bar guard).
        _rows = np.where(np.isin(parse_array, (4, 7)))[0]
        if len(_rows):
            _cap = int(_rows.max() + FIT_BOTTOM_MARGIN.get(fit, 20))
            inpaint_mask[_cap:, :] = 0
    mask = Image.fromarray(inpaint_mask.astype(np.uint8) * 255)
    return mask, Image.fromarray(inpaint_mask.astype(np.uint8) * 127)


SKIN_LABELS = (1, 2, 3, 11, 12, 13, 14, 15, 18)  # hat, hair, sunglasses, head, legs, arms, neck


def restore_skin(orig_img, gen_img, orig_parse, gen_parse):
    """Where BOTH parses say skin -> keep the ORIGINAL camera pixels (bit-exact)."""
    W_, H_ = orig_img.size
    o = np.array(orig_parse.resize((W_, H_), Image.NEAREST))
    g = np.array(gen_parse.resize((W_, H_), Image.NEAREST))
    keep = np.isin(o, SKIN_LABELS) & np.isin(g, SKIN_LABELS)
    keep_f = cv2.GaussianBlur(keep.astype(np.float32), (0, 0), 1.0)[..., None]
    out = (np.array(orig_img).astype(np.float32) * keep_f + np.array(gen_img).astype(np.float32) * (1.0 - keep_f)).astype(np.uint8)
    return Image.fromarray(out)


def composite_final(orig_img, gen_img, inpaint_mask, blur=2.0):
    """Outside the (feathered) garment mask -> ORIGINAL pixels, bit-exact."""
    W_, H_ = orig_img.size
    m = np.array(inpaint_mask.resize((W_, H_), Image.BILINEAR)).astype(np.float32) / 255.0
    m = cv2.GaussianBlur(m, (0, 0), blur)[..., None]
    out = (np.array(orig_img).astype(np.float32) * (1.0 - m) + np.array(gen_img).astype(np.float32) * m).astype(np.uint8)
    return Image.fromarray(out)


GARMENT_LABELS = {'upper': (4, 7), 'lower': (5, 6), 'dress': (4, 5, 6, 7)}


def repair_hem(orig_img, gen_img, orig_parse, gen_parse, inpaint_mask, category='upper', feather=1.5):
    W_, H_ = orig_img.size
    g = np.array(gen_parse.resize((W_, H_), Image.NEAREST))
    labels = GARMENT_LABELS.get(category, (4, 7))
    grows = np.where(np.isin(g, labels))[0]
    if len(grows) == 0:
        return gen_img
    hem = int(np.percentile(grows, 99.5))
    m = np.array(inpaint_mask.resize((W_, H_), Image.NEAREST)) > 128
    below = np.zeros_like(m)
    below[hem:, :] = True
    leftover = m & below & (~np.isin(g, labels))
    if not leftover.any():
        return gen_img
    w = cv2.GaussianBlur(leftover.astype(np.float32), (0, 0), feather)[..., None]
    out = (np.array(orig_img).astype(np.float32) * w + np.array(gen_img).astype(np.float32) * (1.0 - w)).astype(np.uint8)
    return Image.fromarray(out)


# ---------- models ----------
print('loading openpose + parsing ...')
openpose_model = OpenPose(0)
parsing_model = Parsing(0)
print('loading OOTDiffusion-DC (fp16) ...')
ootd_model = OOTDiffusionDC(0)
try:
    ootd_model.pipe.enable_attention_slicing('auto')
    ootd_model.pipe.enable_vae_slicing()
    print('memory slicing: on')
except Exception as e:
    print('memory slicing skipped:', e)
try:
    ootd_model.image_encoder = ootd_model.image_encoder.half()
    ootd_model.text_encoder = ootd_model.text_encoder.half()
    print('encoders: fp16')
except Exception as e:
    print('encoder fp16 skipped:', e)
print('VRAM after load: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))


# ---------- try-on ----------
def tryon(person_path, garment_path, category='upper', fit='regular'):
    t0 = time.time()
    category = (category or 'upper').lower()
    fit = (fit or 'regular').lower()
    person = Image.open(person_path).convert('RGB').resize((W, H))
    garment = Image.open(garment_path).convert('RGB').resize((W, H))
    small = person.resize((384, 512), Image.BILINEAR)
    keypoints = openpose_model(small)
    parse_small, _ = parsing_model(small)
    mask, _ = get_mask_location('dc', CAT_MASK.get(category, 'upper_body'), parse_small, keypoints, fit=fit)
    mask_big = mask.resize((W, H), Image.NEAREST)
    gray_fill = Image.new('RGB', (W, H), (127, 127, 127))
    masked_vton = Image.composite(gray_fill, person, mask_big)
    try:
        out = ootd_model(model_type='dc', category=CAT_MODEL.get(category, 'upperbody'),
                         image_garm=garment, image_vton=masked_vton, mask=mask_big,
                         image_ori=person, num_samples=1, num_steps=STEPS,
                         image_scale=2.0, seed=SEED)[0].resize((W, H))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        raise RuntimeError('GPU ran out of memory. Set HIGH_RES=False at the top of this cell and re-run it.')
    gen_parse, _ = parsing_model(out.resize((384, 512), Image.BILINEAR))
    restored = restore_skin(person, out, parse_small, gen_parse)
    repaired = repair_hem(person, restored, parse_small, gen_parse, mask_big, category)
    final = composite_final(person, repaired, mask_big)
    torch.cuda.empty_cache(); gc.collect()
    print(f'done: cat={category} fit={fit} in {time.time()-t0:.0f}s')
    return final


with gr.Blocks(title='VASTRA Pro Try-On') as demo:
    gr.Markdown('## VASTRA Pro Try-On (OOTDiffusion · mask-accurate · skin-restore)')
    with gr.Row():
        p_in = gr.Image(type='filepath', label='Your photo (full-body)')
        g_in = gr.Image(type='filepath', label='Garment photo')
    with gr.Row():
        c_in = gr.Dropdown(['upper', 'lower', 'dress'], value='upper', label='Garment type')
        f_in = gr.Dropdown(['slim', 'regular', 'loose'], value='regular', label='Fit')
    btn = gr.Button('Try on', variant='primary')
    out = gr.Image(type='pil', label='Result')
    btn.click(fn=tryon, inputs=[p_in, g_in, c_in, f_in], outputs=out, api_name='tryon')
demo.queue().launch(share=True)